# Basic Usage

This page provides an overview of using humancompatible-train for constrained deep learning on a simple example.

## Idea


The core of the package is formed by Lagrangian-based **dual optimizers**, which are PyTorch Optimizer-like objects that handle the **constrained** part of **constrained deep learning**.

They create, keep track of, and update the **dual parameters** of the constrained minimization problem, as well as calculate the Lagrangian that is then minimized by a standard PyTorch optimizer in place of a loss.

Formally, our constrained learning task looks as follows:

$$ \min_{x\in\mathbb{R}^n} \mathbb{E}[f(x,\xi)] \quad \text{s.t.} \quad \mathbb{E}[g(x,\xi)] \leq 0 $$

where $ \xi\sim\Xi $ is a random variable, $ f(x,\xi) $ is the loss function, and $ g(x,\xi) $ is the constraint function. Of course, the loss and constraint functions need not depend on one random variable; $ \xi $ can be treated as a concatenation of several of them.

In this notebook, we shall start with the simpler case of **equality constraints**, i.e. $ \mathbb{E}[g(x,\xi)] = 0 $. You are also welcome to check out the c tutorial.

---

## Simple Example

Let us demonstrate using a **fairness-constrained learning** task, where we want to learn a classifier that is accurate but also satisfies a **demographic parity constraint** - i.e. we would like

$$ | P( Y = 1 | \text{X is Male}) - P ( Y = 1 | \text{X is Female} ) | = \epsilon $$

where $ Y $ is the prediction given by our model for sample $ X $, and $ \epsilon $ is some small threshold.

*Note: Normally, this would be an inequality constraint, but for the sake of this example let us handle this case first.*

---

To enforce demographic parity, we will define a **constraint function** (using the [fairret](https://github.com/aida-ugent/fairret) package) that measures the difference in positive prediction rates between two demographic groups.

The **dual optimizer** will then update the Lagrange multipliers to enforce this constraint during training.

First, let us load and prepare the data. We will use the ACS dataset, containing U.S. Census data, provided by the [folktables](https://github.com/socialfoundations/folktables) package. Feel free to skip this section.

In [ ]:
# load data
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler
from folktables import ACSDataSource, ACSIncome, generate_categories

torch.set_default_dtype(torch.float32)

# load folktables data
data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["FL"], download=True)
definition_df = data_source.get_definitions(download=True)
categories = generate_categories(
    features=ACSIncome.features, definition_df=definition_df
)
df_feat, df_labels, _ = ACSIncome.df_to_pandas(
    acs_data, categories=categories, dummies=True
)
sens_cols = ["SEX_Female", "SEX_Male"]
features = df_feat.drop(columns=sens_cols).to_numpy(dtype=np.float32)
labels = df_labels.to_numpy(dtype=np.float32)
# one-hot encoding of the sensitive attribute (gender)
groups = df_feat[sens_cols].to_numpy(dtype=np.float32)

# standardize features
scaler = StandardScaler()
features = scaler.fit_transform(features)
# convert to torch tensors
X = torch.tensor(features) ; y = torch.tensor(labels) ; groups = torch.tensor(groups)

dataset_train = torch.utils.data.TensorDataset(X, groups, y)
loader = torch.utils.data.DataLoader(dataset_train, batch_size=128, shuffle=True)
criterion = torch.nn.BCEWithLogitsLoss()

Initialize the model and optimizer.

In [ ]:
from torch.nn import Sequential
from torch.optim import AdamW

def setup_model():

    model = Sequential(
        torch.nn.Linear(features.shape[1], 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 32),
        torch.nn.ReLU(),
        torch.nn.Linear(32, 1),
    )

    optimizer = AdamW(model.parameters())
    return model, optimizer

Next, we define the **constraint function** for demographic parity, which uses the `fairret.statistic.PositiveRate` class to evaluate positive rates for both groups.

In [ ]:
from fairret.statistic import PositiveRate

statistic = PositiveRate()

def pr_diff(logit, groups):
    preds = torch.sigmoid(logit)
    stats = PositiveRate()(preds, groups)
    stat_diff = torch.abs(stats[0] - stats[1])
    return stat_diff

As a last step, we define our **dual optimizer**. To set it up, we only need to define the **number of constraints** -- in our case, it is 1 -- so it can create the corresponding dual variables.

In [ ]:
from humancompatible.train.dual_optim import ALM

dual_optimizer = ALM(m=1, lr=0.01, dual_range=(-100, 100), init_duals=0.)

Finally, we write our training loop. In addition to the forward pass and loss calculation, we add a constraint calculation step (0.05 is our $ \epsilon $ threshold).

Then, the `forward_update` step does two things:
- Updates the dual variables based on the constraint violation,
- Calculates the Lagrangian based on loss and constraint violation.

We then perform a backward pass on the Lagrangian and minimize it using a normal PyTorch optimizer.

In [ ]:
epochs = 10

model, optimizer = setup_model()

for epoch in range(epochs):
    # eval
    model.eval()
    logit = model(X)
    train_loss = criterion(logit, y).item()
    train_fair = pr_diff(logit, groups).item()
    print(f"Epoch: {epoch}, loss: {train_loss}, constraint: {train_fair}")
    
    # train
    model.train()
    for batch_feat, batch_groups, batch_label in loader:
        optimizer.zero_grad()
        logit = model(batch_feat)
        loss = criterion(logit, batch_label)
        
        constraint = pr_diff(logit, batch_groups) - 0.05
        lagr = dual_optimizer.forward_update(loss, constraint.unsqueeze(0))
        lagr.backward()
    
        optimizer.step()
        

Epoch: 0, loss: 0.7124184966087341, constraint: 0.0017570257186889648
Epoch: 1, loss: 0.39874470233917236, constraint: 0.06620797514915466
Epoch: 2, loss: 0.39147812128067017, constraint: 0.056648969650268555
Epoch: 3, loss: 0.3884408175945282, constraint: 0.04519534111022949
Epoch: 4, loss: 0.38256096839904785, constraint: 0.046931684017181396
Epoch: 5, loss: 0.37441322207450867, constraint: 0.04030010104179382
Epoch: 6, loss: 0.3695501685142517, constraint: 0.036804407835006714
Epoch: 7, loss: 0.35965678095817566, constraint: 0.03822091221809387
Epoch: 8, loss: 0.354665070772171, constraint: 0.03923347592353821
Epoch: 9, loss: 0.34533509612083435, constraint: 0.03696507215499878


Due to noise, it is difficult to obtain the exact correct value, but the method attempts to keep the constraint around 0.05!

Just in case, let's check what happens if we train the model without constraints:

In [ ]:
model, optimizer = setup_model()

for epoch in range(epochs):
    # eval
    model.eval()
    logit = model(X)
    train_loss = criterion(logit, y).item()
    train_fair = pr_diff(logit, groups).item()
    print(f"Epoch: {epoch}, loss: {train_loss}, constraint: {train_fair}")
    
    # train
    model.train()
    for batch_feat, batch_groups, batch_label in loader:
        optimizer.zero_grad()
        logit = model(batch_feat)
        loss = criterion(logit, batch_label)

        loss.backward()
        optimizer.step()

Epoch: 0, loss: 0.6904992461204529, constraint: 0.0005057156085968018
Epoch: 1, loss: 0.39664506912231445, constraint: 0.08427131175994873
Epoch: 2, loss: 0.38788166642189026, constraint: 0.09328612685203552
Epoch: 3, loss: 0.3807773292064667, constraint: 0.0948910117149353
Epoch: 4, loss: 0.3751002252101898, constraint: 0.09216496348381042
Epoch: 5, loss: 0.3642417788505554, constraint: 0.09508025646209717
Epoch: 6, loss: 0.35605600476264954, constraint: 0.09377643465995789
Epoch: 7, loss: 0.3492068350315094, constraint: 0.09720361232757568
Epoch: 8, loss: 0.3390880823135376, constraint: 0.09501060843467712
Epoch: 9, loss: 0.32931336760520935, constraint: 0.09538483619689941


The absolute difference in positive rates is two times higher than what we wanted!

Further reading: